In [1]:
import pandas as pd
import numpy as np
import os

## Dataset Pedidos

In [2]:
raw_path = os.path.join('..', 'data', 'raw')

In [3]:
path_rjue = os.path.join(raw_path, 'RJUE_Pedidos_de_licenciamento.csv')

df_pedidos = pd.read_csv(path_rjue)
df_pedidos.head()

,Object ID,Código SIG,ID Tipo,Número de Processo,Tipologia,Data de Entrada,Morada,Freguesia,Procedimento,Operação Urbanística,Assunto,FONTE,GlobalID,Shape__Area,Shape__Length
0,26,1501106007001,2.0,10/AE-EDI/2009,Edificação,2009-01-19,R do Norte (Bairro Alto) 48-52,Misericórdia,Sem Informação,Alteração,Sem Informação,GESLIS,d66be4ec-23ff-423f-b160-4b15a9d45115,279.911621,72.624162
1,27,3202209037001,2.0,10/AE-EDI/2010,Edificação,2010-01-22,R de Pedrouços 95-95C,Belém,Sem Informação,Ampliação,Sem Informação,GESLIS,0324c54d-0733-410f-9858-d5145c1bed3a,221.858887,66.782175
2,28,0602601006001,2.0,10/AE-EDI/2011,Edificação,2011-01-14,Av Almirante Reis 24-24B,Arroios,Sem Informação,Ampliação,Sem Informação,GESLIS,eb0b5ec5-719e-44b9-b423-a2ff7ed4f4b1,237.245117,68.767985
3,29,4800109007001,2.0,10/AE-EDI/2012,Edificação,2012-01-31,R dos Sapateiros 175-183,Santa Maria Maior,Sem Informação,Alteração,Alterações Exteriores,GESLIS,ea8861f4-e950-46b5-ac45-c7cbd1503a47,266.523438,65.885287
4,30,4901004010001,2.0,10/AE-EDI/2013,Edificação,2013-02-04,R do Ferragial 25,Misericórdia,Sem Informação,Alteração,Alterações Interiores,GESLIS,ead0b681-c21c-47ea-a253-e8c4912eb7ce,202.484863,58.532356


In [4]:
df_pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31319 entries, 0 to 31318
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Object ID             31319 non-null  int64  
 1   Código SIG            31318 non-null  object 
 2   ID Tipo               31318 non-null  float64
 3   Número de Processo    31319 non-null  object 
 4   Tipologia             31319 non-null  object 
 5   Data de Entrada       31319 non-null  object 
 6   Morada                31319 non-null  object 
 7   Freguesia             31319 non-null  object 
 8   Procedimento          31319 non-null  object 
 9   Operação Urbanística  31319 non-null  object 
 10  Assunto               31319 non-null  object 
 11  FONTE                 31319 non-null  object 
 12  GlobalID              31319 non-null  object 
 13  Shape__Area           31319 non-null  float64
 14  Shape__Length         31319 non-null  float64
dtypes: float64(3), int6

In [5]:
# 1. Limpeza de strings ANTES de filtrar
df_pedidos['Freguesia'] = df_pedidos['Freguesia'].astype(str).str.strip().str.title()

freguesias_lisboa = [
    'Ajuda', 'Alcântara', 'Alvalade', 'Areeiro', 'Arroios', 'Avenidas Novas',
    'Beato', 'Belém','Benfica', 'Campo De Ourique', 'Campolide', 'Carnide',
    'Estrela', 'Lumiar', 'Marvila', 'Misericórdia', 'Olivais',
    'Parque Das Nações', 'Penha De França', 'Santa Clara',
    'Santa Maria Maior', 'Santo António', 'São Domingos De Benfica',
    'São Vicente'
]

# 2. filtro
df_pedidos = df_pedidos[df_pedidos['Freguesia'].isin(freguesias_lisboa)].copy()

# 3. Tratamento de Datas (Especificar o format: format='%Y-%m-%d')
df_pedidos['Data de Entrada'] = pd.to_datetime(df_pedidos['Data de Entrada'], errors='coerce', format='%Y-%m-%d')

# 4. Criar a coluna de Agrupamento
df_pedidos['AnoMes'] = df_pedidos['Data de Entrada'].dt.to_period('M')

In [6]:
df_pedidos['Tipologia'].value_counts()

Tipologia
Edificação    31317
Name: count, dtype: int64

In [7]:
df_pedidos['Procedimento'].value_counts()

Procedimento
Sem Informação    31317
Name: count, dtype: int64

In [8]:
df_pedidos["FONTE"].value_counts()

FONTE
GESLIS    22855
EURBAN     8462
Name: count, dtype: int64

In [9]:
df_pedidos['Operação Urbanística'].unique()

array(['Alteração', 'Ampliação', 'Informação Prévia',
       'Alteração Durante a Execução da Obra', 'Demolição',
       'Obras de Conservação', 'Construção',
       'Renovação de Licença (artigo 72º do RJUE)',
       'Alteração ou Construção para instalação de antenas de radiocomunicações',
       'Op. Urbanísticas promovidas pela Adm. Pública', 'Reconstrução',
       'Dispensa de Requisitos', 'Demolição Autónoma', 'Construção Nova',
       'Não Identificada', 'Construção nova'], dtype=object)

**Alteração Durante a Execução da Obra** (3308 casos): Isto não é uma obra nova. É uma obra que já estava a decorrer e teve uma mudança de projeto. Se o dono está a alterar o projeto a meio, é sinal de que há investimento ativo a acontecer ali ou seja, valorização.
    
- Analysis: These are not new developments but ongoing projects undergoing design changes.
- Significance: This signals active reinvestment. If a developer amends a project midway, it often indicates an upgrade in quality or an adaptation to higher market demands (upscaling). It is a proxy for real-time capital injection and asset appreciation.
- Decision: KEEP. It reflects current market momentum.

**Informação Prévia** (2.441 casos): Pedido ao município para saber se é possível construir ou alterar algo num terreno. Isto é alguém a perguntar à Câmara: "Posso construir aqui?". Muitas vezes a resposta da Câmara é "Não" ou o promotor desiste. Não é obra real, é intenção. Para prever rendas, contar obras reais.

- Analysis: A non-binding inquiry to the Municipality to check if a construction or land-use change is feasible.
- Significance: This is speculative intent, not actual supply. Often, the City Council says "No," or the investor backs out. It does not generate physical construction or immediate rental stock.

**Não Identificada** (806 casos): EXCLUDE

**Op. Urbanísticas promovidas pela Adm. Pública** (283 casos)
- Analysis: Public infrastructure projects such as sidewalk repairs, schools, or community pavilions.
- Significance: While these improve local amenities, they rarely translate into private residential rental stock. Including them would skew the "supply" variable without representing actual apartments entering the market.

**Antenas de radiocomunicações** (108 casos)
- Analysis: Installation of 4G/5G antennas on rooftops.
- Significance: These have zero impact on residential real estate pressure. In some cases, they are considered a negative visual or health externality, potentially decreasing rental value rather than increasing it.

**Renovação de Licença** (76 casos) & **Dispensa de Requisitos** (55)
- Analysis: Purely bureaucratic paperwork. These represent projects that exceeded their original timeline and required an extension.
- Significance: Counting these as "new events" would lead to double-counting. The building is already being tracked; a renewal is just administrative lag (red tape).

In [ ]:
df_pedidos['Operação Urbanística'] = df_pedidos['Operação Urbanística'].astype(str).str.strip().str.title()

mapa_final = {
    # --- GRUPO A: NOVA OFERTA ESTRUTURAL (Aumenta m2 / Fogos) ---
    'Construção': 'PED_Nova_Construcao',
    'Construção Nova': 'PED_Nova_Construcao',
    
    # --- GRUPO B: VALORIZAÇÃO DO STOCK (Gentrificação) ---
    'Alteração': 'PED_Reabilitacao',
    'Reconstrução': 'PED_Reabilitacao',
    'Obras De Conservação': 'PED_Reabilitacao',
    'Ampliação': 'PED_Reabilitacao',
    "Alteração Durante A Execução Da Obra": 'PED_Reabilitacao',

    # --- GRUPO C: DEMOLIÇÃO (Opcional, mas vamos manter separado) ---
    'Demolição': 'PED_Demolicao',
    'Demolição Autónoma': 'PED_Demolicao'

    # TUDO O RESTO SERÁ MAPEADO PARA NaN E REMOVIDO
}

df_pedidos['Categoria_PED'] = df_pedidos['Operação Urbanística'].map(mapa_final)

lixo = df_pedidos[df_pedidos['Categoria_PED'].isna()]['Operação Urbanística'].value_counts()
df_validos = df_pedidos.dropna(subset=['Categoria_PED']).copy()

In [10]:
# 1. Identificar Duplicados Totais (Linhas exatamente iguais)
duplicados_totais = df_validos.duplicated().sum()

# 2. Identificar Duplicados de Negócio (O mesmo projeto no mesmo mês)
chaves_negocio = ['Código SIG', 'Freguesia', 'Shape__Area', 'Categoria_PED','Shape__Length', 'AnoMes', "ID Tipo"]
duplicados_negocio = df_validos.duplicated(subset=chaves_negocio).sum()

print(f"--- RELATÓRIO DE DUPLICADOS ---")
print(f"Duplicados exatos (linhas idênticas): {duplicados_totais}")
print(f"Duplicados de negócio (mesmo projeto/mês): {duplicados_negocio}")


# Manter apenas a primeira ocorrência
df_pedidos_final = df_validos.drop_duplicates(subset=chaves_negocio, keep='first')

print(f"\n✅ Dataset limpo. Registos restantes: {len(df_validos)}")

--- RELATÓRIO DE DUPLICADOS ---
Duplicados exatos (linhas idênticas): 0
Duplicados de negócio (mesmo projeto/mês): 588

✅ Dataset limpo. Registos restantes: 27548


In [11]:
# 1. Agregação Mensal: Somar Área e Contar Pedidos por Freguesia/Mês/Categoria
df_mensal = df_pedidos_final.groupby(['Freguesia', 'AnoMes', 'Categoria_PED']).agg(
    Area_Mensal=('Shape__Area', 'sum'),
    Count_Mensal=('Categoria_PED', 'count')
).reset_index()

# 1. Antes do Pivot, garante que o AnoMes no df_mensal é String
df_mensal['AnoMes'] = df_mensal['AnoMes'].astype(str)

# 2. Pivot Table: Transformar as Categorias em Colunas (Nova_Construcao, Reabilitacao, etc.)
df_pivot = df_mensal.pivot_table(
    index=['Freguesia', 'AnoMes'], 
    columns='Categoria_PED', 
    values=['Area_Mensal', 'Count_Mensal'],
    fill_value=0
).reset_index()

# Achatar o nome das colunas (ex: ('Area_Mensal', 'PED_Nova_Construcao') -> 'PED_NovaConstrucao_Area')
df_pivot.columns = [f"{col[1]}_{col[0]}".replace('_Mensal', '') if col[1] else col[0] for col in df_pivot.columns]

# 3. Criar o Esqueleto Temporal DINÂMICO
freguesias = df_pedidos_final['Freguesia'].unique()
min_data = df_mensal['AnoMes'].min()
max_data = df_mensal['AnoMes'].max()

datas_dinamicas = pd.period_range(start=min_data, end=max_data, freq='M').astype(str)
esqueleto = pd.MultiIndex.from_product([freguesias, datas_dinamicas], names=['Freguesia', 'AnoMes']).to_frame(index=False)

# 4. Merge com o Esqueleto e preencher vazios com 0
df_final_pedidos = pd.merge(esqueleto, df_pivot, on=['Freguesia', 'AnoMes'], how='left').fillna(0)

# 5. CÁLCULO DO STOCK ACUMULADO (Rolling 12 Months)
# Isto é o que o Random Forest realmente vai usar para prever o preço
cols_para_acumular = [c for c in df_final_pedidos.columns if 'PED_' in c]

for col in cols_para_acumular:
    df_final_pedidos[f'Stock_12M_{col}'] = (
        df_final_pedidos.groupby('Freguesia')[col]
        .transform(lambda x: x.rolling(window=12, min_periods=1).sum())
        .shift(1)
    )

print(f"✅ Agregação concluída: {len(df_final_pedidos)} linhas (24 Freguesias x 112 Meses)")

✅ Agregação concluída: 4896 linhas (24 Freguesias x 112 Meses)


Real estate prices are lagging indicators, they reflect the final result of months of market pressure. Conversely, urban planning permits and licenses are leading indicators, they signal future supply and investment intent.
- The .shift(1) Methodology: To prevent Data Leakage, we applied a one-month temporal shift ($t-1$) to all urban features.
- Justification: A rental price recorded in May cannot be influenced by a permit issued later that same month. Therefore, the price for Month $M$ is explained by the accumulated urban stock as of the last day of Month $M-1$. This ensures the model only "knows" facts that were already established before the month began.

**Gentrification Ratio**: Beyond physical construction, the model captures Market Expectation. Prices often rise based on the "smoke" of intent—permits being filed—well before a project is completed.

- Ratio $> 1$: Focus on upgrading existing stock (Classic Gentrification). The neighborhood is being "reclaimed" and premiumized.
- Ratio $< 1$: Focus on growth via expansion (Urban Sprawl/Expansion). The city is growing "outward.
- "High Ratio (e.g., $10.0$): Typical of Historical Districts (e.g., Misericórdia, Santa Maria Maior) where there is no physical space for new builds, only for the deep renovation of heritage assets.

A. The Area Ratio ($m^2$): 
- Insight: This measures the volume of premiumization.
- Market Signal: A high area ratio (e.g., a single $10,000$ $m^2$ palace renovation) indicates a massive shift in the available square footage toward high-end segments. It signals a "Mass Upgrade" of the local offer.

B. The Count Ratio:
- Insight: This measures the frequency of investment intent.
- Market Signal: High counts (e.g., 50 small renovation requests in 50 different buildings) are the purest indicator of Social Gentrification. It demonstrates a widespread, decentralized movement of valuation. The market "feels" the entire neighborhood is changing, which exerts a more persistent pressure on rental prices than a single large-scale project.

In [13]:
# 1. Cálculo do Rácio de Gentrificação para as Intenções (Pedidos)
# Rácio = Área de Reabilitação / Área de Nova Construção
# Usamos o Stock de 12 meses porque é mais estável

# Definir as colunas de stock que criámos no passo anterior
col_reab = 'Stock_12M_PED_Reabilitacao_Area'
col_nova = 'Stock_12M_PED_Nova_Construcao_Area'

# Calcular o rácio
# Adicionamos um valor ínfimo (epsilon) no denominador para evitar divisões por zero
epsilon = 0.0001 
df_final_pedidos['Racio_Gentrificacao_PED_Area'] = (
    df_final_pedidos[col_reab] / (df_final_pedidos[col_nova] + epsilon)
)

# 2. Rácio baseado em Contagem (Opcional, mas útil)
# Indica se há "muitos pequenos projetos" vs "poucos grandes prédios"
col_reab_cnt = 'Stock_12M_PED_Reabilitacao_Count'
col_nova_cnt = 'Stock_12M_PED_Nova_Construcao_Count'

df_final_pedidos['Racio_Gentrificacao_PED_Count'] = (
    df_final_pedidos[col_reab_cnt] / (df_final_pedidos[col_nova_cnt] + epsilon)
)

# 3. Limpeza de valores infinitos (caso existam)
df_final_pedidos = df_final_pedidos.replace([np.inf, -np.inf], 0)

print("✅ Rácios de Gentrificação (Intenção) calculados com sucesso.")
print(df_final_pedidos[['Freguesia', 'AnoMes', 'Racio_Gentrificacao_PED_Area']].tail())

✅ Rácios de Gentrificação (Intenção) calculados com sucesso.
     Freguesia   AnoMes  Racio_Gentrificacao_PED_Area
4891    Lumiar  2025-08                      2.943009
4892    Lumiar  2025-09                      3.574977
4893    Lumiar  2025-10                      3.580063
4894    Lumiar  2025-11                      3.569263
4895    Lumiar  2025-12                      4.673400


In [14]:
df_final_pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4896 entries, 0 to 4895
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Freguesia                            4896 non-null   object 
 1   AnoMes                               4896 non-null   object 
 2   PED_Demolicao_Area                   4896 non-null   float64
 3   PED_Nova_Construcao_Area             4896 non-null   float64
 4   PED_Reabilitacao_Area                4896 non-null   float64
 5   PED_Demolicao_Count                  4896 non-null   float64
 6   PED_Nova_Construcao_Count            4896 non-null   float64
 7   PED_Reabilitacao_Count               4896 non-null   float64
 8   Stock_12M_PED_Demolicao_Area         4895 non-null   float64
 9   Stock_12M_PED_Nova_Construcao_Area   4895 non-null   float64
 10  Stock_12M_PED_Reabilitacao_Area      4895 non-null   float64
 11  Stock_12M_PED_Demolicao_Count 

## Dataset Alvarás

In [ ]:
path_rjue_alvaras = os.path.join(raw_path, 'RJUE_Alvaras.csv')

In [ ]:
df_alvaras = pd.read_csv(path_rjue_alvaras)

In [16]:
df_alvaras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11813 entries, 0 to 11812
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Object ID             11813 non-null  int64  
 1   Código SIG            11813 non-null  int64  
 2   Número de Processo    11813 non-null  object 
 3   Data de Entrada       11813 non-null  object 
 4   Morada                11813 non-null  object 
 5   Freguesia             11813 non-null  object 
 6   Operação Urbanística  11813 non-null  object 
 7   Assunto               11813 non-null  object 
 8   Procedimento          11813 non-null  object 
 9   Tipo de Alvará        11813 non-null  object 
 10  Número de Alvará      11813 non-null  object 
 11  Data de Alvará        11813 non-null  object 
 12  ID Tipo               11813 non-null  int64  
 13  Tipologia             11813 non-null  object 
 14  FONTE                 11813 non-null  object 
 15  GlobalID           

- Data de Entrada: Quando pediram para fazer a obra (pode ter sido em 2018).
- Data de Alvará: Quando a Câmara emitiu o papel e a obra pode arrancar (pode ser 2020).
Para esta base de dados, ignoramos a Data de Entrada. O impacto no mercado (a certeza de obra) acontece na emissão. Se usar a data de entrada, vou estar a dizer ao modelo que a obra começou 2 anos antes de ter autorização.

In [17]:
df_alvaras['Operação Urbanística'].value_counts()

Operação Urbanística
Alteração                                                                  4856
Ampliação                                                                  3470
Alteração Durante a Execução da Obra                                       2055
Construção                                                                  781
Demolição                                                                   419
Demolição Autónoma                                                           72
Obras de Conservação                                                         59
Reconstrução                                                                 39
Alteração ou Construção para instalação de antenas de radiocomunicações      38
Renovação de Licença (artigo 72º do RJUE)                                    19
Operações Isentas de Licenciamento ou Autorização                             5
Name: count, dtype: int64

In [18]:
df_alvaras['Tipo de Alvará'].value_counts()

Tipo de Alvará
EO                4774
ADT               1800
OD                1742
CPREV             1192
AE-EO              797
ED                 478
CE                 421
CD                 329
AE-CPREV           140
Sem Informação     109
AE-CE               25
                     3
AE-ADT               2
AE-ED                1
Name: count, dtype: int64

In [19]:
df_alvaras.columns = [
    'Object_ID', 'Cod_SIG', 'N_Processo', 'Data_Entrada', 'Morada', 
    'Freguesia', 'Operacao_Urbanistica', 'Assunto', 'Procedimento', 
    'Tipo_Alvara', 'N_Alvara', 'Data_Alvara', 'ID_Tipo', 'Tipologia', 
    'Fonte', 'GlobalID', 'Shape_Area', 'Shape_Length'
]

# 2. Filtro Geográfico e Normalização
df_alvaras['Freguesia'] = df_alvaras['Freguesia'].astype(str).str.strip().str.title()
freguesias_lisboa = [
    'Ajuda', 'Alcântara', 'Alvalade', 'Areeiro', 'Arroios', 'Avenidas Novas',
    'Beato', 'Belém','Benfica', 'Campo De Ourique', 'Campolide', 'Carnide',
    'Estrela', 'Lumiar', 'Marvila', 'Misericórdia', 'Olivais',
    'Parque Das Nações', 'Penha De França', 'Santa Clara',
    'Santa Maria Maior', 'Santo António', 'São Domingos De Benfica',
    'São Vicente'
]
df_alvaras = df_alvaras[df_alvaras['Freguesia'].isin(freguesias_lisboa)].copy()

In [20]:
# Tens de usar a Data de ALVARÁ, não a de Entrada.
df_alvaras['Data_Alvara'] = pd.to_datetime(df_alvaras['Data_Alvara'], errors='coerce')

# Criar o AnoMes com base na emissão do alvará
df_alvaras['AnoMes'] = df_alvaras['Data_Alvara'].dt.to_period('M')

In [21]:
mapa_alvaras = {
    # Grupo Nova Oferta
    'Construção': 'ALV_Nova_Construcao',

    # Grupo Reabilitação
    'Ampliação': 'ALV_Reabilitacao',
    'Alteração': 'ALV_Reabilitacao',
    'Reconstrução': 'ALV_Reabilitacao',
    'Obras de Conservação': 'ALV_Reabilitacao',
    'Alteração Durante a Execução da Obra': 'ALV_Reabilitacao',
    
    # Grupo Demolição
    'Demolição': 'ALV_Demolicao',
    'Demolição Autónoma': 'ALV_Demolicao'
}

df_alvaras['Categoria_ALV'] = df_alvaras['Operacao_Urbanistica'].map(mapa_alvaras)
df_alvaras_limpo = df_alvaras.dropna(subset=['Categoria_ALV']).copy()

In [22]:
# Usamos o 'Data de Alvará' convertido para AnoMes
df_alvaras_limpo['AnoMes'] = pd.to_datetime(df_alvaras_limpo['Data_Alvara']).dt.to_period('M').astype(str)
chaves_alv = [
    'Cod_SIG', 
    'Freguesia', 
    'AnoMes', 
    'Shape_Area', 
    'Shape_Length', 
    'Categoria_ALV',
    "ID_Tipo"
]

# Limpeza
antes = len(df_alvaras_limpo)
df_alvaras_final = df_alvaras_limpo.drop_duplicates(subset=chaves_alv, keep='first')
depois = len(df_alvaras_final)

print(f"🔥 Operação concluída: {antes - depois} duplicados de negócio eliminados.")

🔥 Operação concluída: 104 duplicados de negócio eliminados.


In [23]:
# AGREGAÇÃO MENSAL
df_alv_mensal = df_alvaras_final.groupby(['Freguesia', 'AnoMes', 'Categoria_ALV']).agg(
    Area_Mensal=('Shape_Area', 'sum'),
    Count_Mensal=('Categoria_ALV', 'count')
).reset_index()

#ANO MES COMO STRING
df_alv_mensal['AnoMes'] = df_alv_mensal['AnoMes'].astype(str)

# PIVOT E ESQUELETO (Sincronização com o horizonte temporal dos Pedidos)
df_alv_pivot = df_alv_mensal.pivot_table(
    index=['Freguesia', 'AnoMes'],
    columns='Categoria_ALV',
    values=['Area_Mensal', 'Count_Mensal'],
    fill_value=0
).reset_index()

# Achatar nomes: ALV_Nova_Construcao_Area, etc.
df_alv_pivot.columns = [f"{col[1]}_{col[0]}".replace('_Mensal', '') if col[1] else col[0] for col in df_alv_pivot.columns]

# Merge com o esqueleto para garantir 0s nos meses vazios
df_final_alvaras = pd.merge(esqueleto, df_alv_pivot, on=['Freguesia', 'AnoMes'], how='left').fillna(0)

# 6. STOCK ACUMULADO (Rolling 12M + Shift 1)
cols_alv = [c for c in df_final_alvaras.columns if 'ALV_' in c]
for col in cols_alv:
    df_final_alvaras[f'Stock_12M_{col}'] = (
        df_final_alvaras.groupby('Freguesia')[col]
        .transform(lambda x: x.rolling(window=12, min_periods=12).sum())
        .shift(1)
    )

# 7. RÁCIO DE GENTRIFICAÇÃO (Fogo Real)
epsilon = 0.0001
df_final_alvaras['Racio_Gentrificacao_ALV_Area'] = (
    df_final_alvaras['Stock_12M_ALV_Reabilitacao_Area'] / 
    (df_final_alvaras['Stock_12M_ALV_Nova_Construcao_Area'] + epsilon)
)

# Rácio de Count: A pulsação de obras no bairro
df_final_alvaras['Racio_Gentrificacao_ALV_Count'] = (
    df_final_alvaras['Stock_12M_ALV_Reabilitacao_Count'] / 
    (df_final_alvaras['Stock_12M_ALV_Nova_Construcao_Count'] + epsilon)
)

print(f"✅ Dataset de Alvarás pronto: {len(df_final_alvaras)} linhas.")

✅ Dataset de Alvarás pronto: 4896 linhas.


In [24]:
# 1. Definir os limites temporais (A tua nova janela de análise)
data_inicio = '2016-08'
data_fim = '2025-11'

# 2. Cortar o dataset dos Pedidos
dataset_pedidos_final = df_final_pedidos[
    (df_final_pedidos['AnoMes'] >= data_inicio) & 
    (df_final_pedidos['AnoMes'] <= data_fim)
].copy()

# 3. Cortar o dataset dos Alvarás
dataset_alvaras_final = df_final_alvaras[
    (df_final_alvaras['AnoMes'] >= data_inicio) & 
    (df_final_alvaras['AnoMes'] <= data_fim)
].copy()

# 4. Diagnóstico de Sanidade
print(f"--- DIAGNÓSTICO DE FILTRAGEM ---")
print(f"Pedidos: {len(dataset_pedidos_final)} linhas | NaNs de Stock: {dataset_pedidos_final.filter(like='Stock').isna().sum().sum()}")
print(f"Alvarás: {len(dataset_alvaras_final)} linhas | NaNs de Stock: {dataset_alvaras_final.filter(like='Stock').isna().sum().sum()}")

--- DIAGNÓSTICO DE FILTRAGEM ---
Pedidos: 2688 linhas | NaNs de Stock: 0
Alvarás: 2688 linhas | NaNs de Stock: 0


In [25]:
dataset_pedidos_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2688 entries, 91 to 4894
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Freguesia                            2688 non-null   object 
 1   AnoMes                               2688 non-null   object 
 2   PED_Demolicao_Area                   2688 non-null   float64
 3   PED_Nova_Construcao_Area             2688 non-null   float64
 4   PED_Reabilitacao_Area                2688 non-null   float64
 5   PED_Demolicao_Count                  2688 non-null   float64
 6   PED_Nova_Construcao_Count            2688 non-null   float64
 7   PED_Reabilitacao_Count               2688 non-null   float64
 8   Stock_12M_PED_Demolicao_Area         2688 non-null   float64
 9   Stock_12M_PED_Nova_Construcao_Area   2688 non-null   float64
 10  Stock_12M_PED_Reabilitacao_Area      2688 non-null   float64
 11  Stock_12M_PED_Demolicao_Count     

In [26]:
dataset_alvaras_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2688 entries, 91 to 4894
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Freguesia                            2688 non-null   object 
 1   AnoMes                               2688 non-null   object 
 2   ALV_Demolicao_Area                   2688 non-null   float64
 3   ALV_Nova_Construcao_Area             2688 non-null   float64
 4   ALV_Reabilitacao_Area                2688 non-null   float64
 5   ALV_Demolicao_Count                  2688 non-null   float64
 6   ALV_Nova_Construcao_Count            2688 non-null   float64
 7   ALV_Reabilitacao_Count               2688 non-null   float64
 8   Stock_12M_ALV_Demolicao_Area         2688 non-null   float64
 9   Stock_12M_ALV_Nova_Construcao_Area   2688 non-null   float64
 10  Stock_12M_ALV_Reabilitacao_Area      2688 non-null   float64
 11  Stock_12M_ALV_Demolicao_Count     

As colunas de área e contagem mensais (ex: PED_Demolicao_Area, ALV_Nova_Construcao_Count) são extremamente voláteis.
- Num mês tens 10 pedidos, no outro tens 0.
- Para um modelo de preços, este "ziguezague" é apenas ruído estatístico que não explica a tendência de longo prazo do mercado.

O que realmente "move o ponteiro" são os Stocks de 12 Meses e os Rácios.
- Stocks 12M: Representam a pressão acumulada e suavizada. É o que o investidor e o comprador sentem no bairro.
- Rácios de Gentrificação: Eles dizem ao modelo que tipo de transformação está a acontecer.

In [27]:
# Definir apenas as colunas que interessam para o modelo final
cols_ped = [
    'Freguesia', 'AnoMes',
    'Stock_12M_PED_Nova_Construcao_Area', 'Stock_12M_PED_Reabilitacao_Area',
    'Stock_12M_PED_Nova_Construcao_Count', 'Stock_12M_PED_Reabilitacao_Count',
    'Stock_12M_PED_Demolicao_Area', 'Stock_12M_PED_Demolicao_Count',
    'Racio_Gentrificacao_PED_Area', 'Racio_Gentrificacao_PED_Count'
]

cols_alv = [
    'Freguesia', 'AnoMes',
    'Stock_12M_ALV_Nova_Construcao_Area', 'Stock_12M_ALV_Reabilitacao_Area',
    'Stock_12M_ALV_Nova_Construcao_Count', 'Stock_12M_ALV_Reabilitacao_Count',
    'Stock_12M_ALV_Demolicao_Area', 'Stock_12M_ALV_Demolicao_Count',
    'Racio_Gentrificacao_ALV_Area', 'Racio_Gentrificacao_ALV_Count'
]

df_pedidos_ready= dataset_pedidos_final[cols_ped]
df_alvaras_ready = dataset_alvaras_final[cols_alv]

In [28]:
## exclude Santa Clara, Beato, Ajuda, olivais, Alcântara, Marvila, Carnide
exclude_fregs = ['Santa Clara', 'Beato', 'Ajuda', 'Olivais', 'Alcântara', 'Marvila', 'Carnide']
df_pedidos_ready = df_pedidos_ready[~df_pedidos_ready['Freguesia'].isin(exclude_fregs)].copy()  
df_alvaras_ready = df_alvaras_ready[~df_alvaras_ready['Freguesia'].isin(exclude_fregs)].copy()

In [ ]:
output_dir = os.path.join('..', 'data', 'processed')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

path_alvaras = os.path.join(output_dir, 'RJUE_dados_urbanismo_alvaras.csv')
df_alvaras_ready.to_csv(path_alvaras, index=False, encoding='utf-8-sig')

path_pedidos = os.path.join(output_dir, 'RJUE_dados_urbanismo_pedidos.csv')
df_pedidos_ready.to_csv(path_pedidos, index=False, encoding='utf-8-sig')